# Scrapping actress names 
from [here](https://talent-dictionary.com/s/jobs/3)

In [75]:
import numpy as np
import pandas as pd
import urllib.request, urllib.parse, urllib.error
from bs4 import BeautifulSoup 

In [82]:
# age10s = np.arange(10, 50+10, 10)
age10s = np.arange(60, 90+10, 10)
df_girls = pd.DataFrame()

Get page nos. for the category

In [83]:
def get_n_page(service_url):

    uh = urllib.request.urlopen(service_url)

    html = ''
    if uh.status==200:   html = uh.read().decode()

    soup = BeautifulSoup(html, 'html.parser')
    soup_lastpage = soup.find('span', attrs={'class':'last'})
    url_lastpage = soup_lastpage.a.get('href')
    
    parsed_url = urllib.parse.urlparse(url_lastpage)
    n_page_str = urllib.parse.parse_qs(parsed_url.query)['page'][0]
    n_page = int(n_page_str)
    
    return n_page

for each 10-age group, get each of name and age

In [84]:
for age10 in age10s:
    service_url = f'https://talent-dictionary.com/s/jobs/3/{age10}'

    n_page = get_n_page(service_url)

    pages = np.arange(1, n_page+1)
    for page in pages:

        params = {'page': page}
        url = service_url + '?' + urllib.parse.urlencode(params)

        print('Retrieving', url)
        uh = urllib.request.urlopen(url)

        html = ''
        if uh.status==200:   html = uh.read().decode()
        print('Retrieved', len(html), 'characters')

        soup = BeautifulSoup(html, 'html.parser')
        soup_girls = soup.find_all("div", attrs={"class": "home_talent_list_wrapper"})[0].ul.find_all('li')

        for soup_girl in soup_girls:
            try: 
                name = soup_girl.find(attrs={'class':'title'}).text
            except:
                name = 'Unknown'
                
            try:
                age = np.float(soup_girl.find(attrs={'class':'age'}).text)
            except:
                age = age10 + 5
                
            
            df_girls = df_girls.append({'name': name, 'age': age}, ignore_index=True)

Retrieving https://talent-dictionary.com/s/jobs/3/60?page=1
Retrieved 41918 characters
Retrieving https://talent-dictionary.com/s/jobs/3/60?page=2
Retrieved 42007 characters
Retrieving https://talent-dictionary.com/s/jobs/3/60?page=3
Retrieved 41515 characters
Retrieving https://talent-dictionary.com/s/jobs/3/60?page=4
Retrieved 41478 characters
Retrieving https://talent-dictionary.com/s/jobs/3/60?page=5
Retrieved 42241 characters
Retrieving https://talent-dictionary.com/s/jobs/3/60?page=6
Retrieved 43026 characters
Retrieving https://talent-dictionary.com/s/jobs/3/60?page=7
Retrieved 42583 characters
Retrieving https://talent-dictionary.com/s/jobs/3/60?page=8
Retrieved 43078 characters
Retrieving https://talent-dictionary.com/s/jobs/3/60?page=9
Retrieved 43513 characters
Retrieving https://talent-dictionary.com/s/jobs/3/60?page=10
Retrieved 43377 characters
Retrieving https://talent-dictionary.com/s/jobs/3/60?page=11
Retrieved 42944 characters
Retrieving https://talent-dictionary.com/

# Extracting Surname and First name
https://github.com/rskmoi/namedivider-python

In [90]:
from namedivider import NameDivider
divider = NameDivider()
divided_name = divider.divide_name("菅義偉")

In [91]:
divided_name

DividedName(family='菅', given='義偉', separator=' ', score=0.6328842762252201, algorithm='kanji_feature')

In [92]:
print(df_actress.head())
print(df_actress.shape)

    age   name
0  17.0   鈴木梨央
1  19.0  石井萌々果
2  16.0   大川春菜
3  11.0   新津ちせ
4  19.0   畑中有里
(5015, 2)


Name should be at least 3 characters

In [99]:
df_actress = df_actress.loc[df_actress.name.apply(lambda x: len(x))>2]

In [100]:
df_actress['family'] = df_actress.name.apply(lambda x: divider.divide_name(x).family)
df_actress['given'] = df_actress.name.apply(lambda x: divider.divide_name(x).given)

<ipython-input-100-81143d2c8f28>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_actress['family'] = df_actress.name.apply(lambda x: divider.divide_name(x).family)
<ipython-input-100-81143d2c8f28>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_actress['given'] = df_actress.name.apply(lambda x: divider.divide_name(x).given)


# Output

In [102]:
df_actress.to_csv('df_actress.csv')